# Day 11 — Fine-tune DistilBERT on IMDB sentiment
Requires: `pip install transformers datasets accelerate torch`


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np, torch

ds = load_dataset('imdb')
tok = AutoTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize(b): return tok(b['text'], truncation=True, max_length=256)
ds_tok = ds.map(tokenize, batched=True)
ds_tok = ds_tok.rename_column('label', 'labels')
ds_tok.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': (preds == labels).mean()}

args = TrainingArguments(
    output_dir='out-distilbert-imdb',
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to='none',
)
trainer = Trainer(
    model=model, args=args,
    train_dataset=ds_tok['train'].shuffle(seed=0).select(range(20000)),
    eval_dataset=ds_tok['test'].select(range(5000)),
    tokenizer=tok,
    compute_metrics=compute_metrics,
)
trainer.train()
print(trainer.evaluate())


## Inference on your own sentences

In [ ]:
from transformers import pipeline
clf = pipeline('text-classification', model=trainer.model, tokenizer=tok, device=0 if torch.cuda.is_available() else -1)
for s in [
    'absolutely loved this movie, the acting was superb',
    'a complete waste of two hours, terrible script',
    'not bad, but I was expecting more',
]:
    print(s, '->', clf(s)[0])
